# Setup

In [16]:
library(tidyverse)
library(here)
library(GenomicRanges)
library(rtracklayer)
here::i_am("figures/SuppFig7_new/PanelB/PanelB.ipynb")

here() starts at /mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics



# load data

In [17]:
ifn_PRO <- read.delim(
    here("inputs","countmat","PROseq_eRNA","IFN_v_Veh_PROseq_cRE.tsv"), sep='\t')
ens <- read.delim(
    here("inputs","ENSEMBL","ensembl.canonical.prot-coding.transcript.tsv"), sep='\t')

deg <- read.delim(
    here("inputs","DEGs", "GM12878", "BAFinh_stimulus","PROseq", "PRO_DEGs_IFN_Veh.tsv"), sep='\t')
counts <- read.delim(
    here("inputs","DEGs","GM12878","BAFinh_stimulus","PROseq","PRO_counts_genes.tsv"), sep='\t')

dREG_veh <- import("/mnt/disk/home/agulka/projects/PROseq_IFN/PROseq_GM12878_IFN/dreg/veh/veh.dREG.peak.full.bed.gz", format="BED", extraCols = c(dreg_score = "numeric", pvalue = "numeric", summit = "numeric"))
dREG_ifn <- import("/mnt/disk/home/agulka/projects/PROseq_IFN/PROseq_GM12878_IFN/dreg/ifn/ifn.dREG.peak.full.bed.gz", format="BED", extraCols = c(dreg_score = "numeric", pvalue = "numeric", summit = "numeric"))


# annotate cREs based on dREG overlaps and whether or not they are located within transcribed protein-coding genes

In [25]:

GENE_CPM <- 1   # veh_IFN CPM at or above which a host gene counts as transcribed
cpm  <- apply(counts, 2, function(x) 1000000*x/sum(x))
veh_cpm <- rowMeans(cpm[, grepl("veh_veh", colnames(cpm))])
ifn_cpm <- rowMeans(cpm[, grepl("veh_IFN", colnames(cpm))])
txn_veh  <- deg$ENSEMBL_ID[veh_cpm >= GENE_CPM]
txn_ifn  <- deg$ENSEMBL_ID[ifn_cpm >= GENE_CPM]

gr_veh_txn<- GRanges(paste0("chr", ens$Chromosome.scaffold.name),
                  IRanges(ens$Transcript.start..bp., ens$Transcript.end..bp.))[
                  ens$Gene.stable.ID %in% txn_veh]

gr_ifn_txn<- GRanges(paste0("chr", ens$Chromosome.scaffold.name),
                  IRanges(ens$Transcript.start..bp., ens$Transcript.end..bp.))[
                  ens$Gene.stable.ID %in% txn_ifn]

dat <- ifn_PRO
dat$inside_veh <- overlapsAny(GRanges(ifn_PRO$seqnames, IRanges(ifn_PRO$start, ifn_PRO$end)), gr_veh_txn)
dat$inside_ifn <- overlapsAny(GRanges(ifn_PRO$seqnames, IRanges(ifn_PRO$start, ifn_PRO$end)), gr_ifn_txn)

annot_gr <- makeGRangesFromDataFrame(ifn_PRO)
ol_veh <- findOverlaps(annot_gr, dREG_veh)
ol_ifn <- findOverlaps(annot_gr, dREG_ifn)

dat$dREG_veh_ol <- 0
dat$dREG_veh_ol[queryHits(ol_veh)] <- 1
dat$dREG_ifn_ol <- 0
dat$dREG_ifn_ol[queryHits(ol_ifn)] <- 1

# write out BED files for heatmap plotting

In [26]:
bed1 <- dat %>% 
    filter(grepl("active_enh", type)) %>%
    filter(inside_veh==TRUE) %>% 
    filter(dREG_veh_ol==TRUE) %>%
    select(seqnames,start,end)

bed2 <- dat %>% 
    filter(grepl("active_enh", type)) %>%
    filter(inside_veh==TRUE) %>% 
    filter(dREG_veh_ol==FALSE) %>%
    select(seqnames,start,end)

bed3 <- dat %>% 
    filter(grepl("active_enh", type)) %>%
    filter(inside_veh==FALSE) %>% 
    filter(dREG_veh_ol==TRUE) %>%
    select(seqnames, start, end)

bed4 <- dat %>% 
    filter(grepl("active_enh", type)) %>%
    filter(inside_veh==FALSE) %>% 
    filter(dREG_veh_ol==FALSE) %>%
    select(seqnames, start, end)

In [27]:
bed5 <- dat %>% 
    filter(grepl("active_enh", type)) %>%
    filter(inside_ifn==TRUE) %>% 
    filter(dREG_ifn_ol==TRUE) %>%
    select(seqnames,start,end)

bed6 <- dat %>% 
    filter(grepl("active_enh", type)) %>%
    filter(inside_ifn==TRUE) %>% 
    filter(dREG_ifn_ol==FALSE) %>%
    select(seqnames,start,end)

bed7 <- dat %>% 
    filter(grepl("active_enh", type)) %>%
    filter(inside_ifn==FALSE) %>% 
    filter(dREG_ifn_ol==TRUE) %>%
    select(seqnames, start, end)

bed8 <- dat %>% 
    filter(grepl("active_enh", type)) %>%
    filter(inside_ifn==FALSE) %>% 
    filter(dREG_ifn_ol==FALSE) %>%
    select(seqnames, start, end)

In [28]:
write.table(bed1, "veh_inGene_dREG.bed", col.names=F, row.names=F, quote=F, sep='\t')
write.table(bed2, "veh_inGene_nodREG.bed", col.names=F, row.names=F, quote=F, sep='\t')
write.table(bed3, "veh_NotinGene_dREG.bed", col.names=F, row.names=F, quote=F, sep='\t')
write.table(bed4, "veh_NotinGene_nodREG.bed", col.names=F, row.names=F, quote=F, sep='\t')
write.table(bed5, "ifn_inGene_dREG.bed", col.names=F, row.names=F, quote=F, sep='\t')
write.table(bed6, "ifn_inGene_nodREG.bed", col.names=F, row.names=F, quote=F, sep='\t')
write.table(bed7, "ifn_NotinGene_dREG.bed", col.names=F, row.names=F, quote=F, sep='\t')
write.table(bed8, "ifn_NotinGene_nodREG.bed", col.names=F, row.names=F, quote=F, sep='\t')

In [35]:
str(bed5)
str(bed6)
str(bed7)
str(bed8)

'data.frame':	4688 obs. of  3 variables:
 $ seqnames: chr  "chr1" "chr1" "chr1" "chr1" ...
 $ start   : int  1677519 2234208 3656053 3667196 3673342 3676535 6290275 6860939 7403304 8395368 ...
 $ end     : int  1679188 2235079 3657411 3667707 3674608 3677852 6291627 6862418 7403980 8395912 ...
'data.frame':	6694 obs. of  3 variables:
 $ seqnames: chr  "chr1" "chr1" "chr1" "chr1" ...
 $ start   : int  1418617 1828207 1841482 1842319 1847725 1862982 1864347 2255399 2290122 3664193 ...
 $ end     : int  1419457 1829861 1842018 1843368 1848543 1863641 1865059 2256665 2291598 3665188 ...
'data.frame':	5038 obs. of  3 variables:
 $ seqnames: chr  "chr1" "chr1" "chr1" "chr1" ...
 $ start   : int  1417873 1602393 1607163 1907471 2642402 4796244 6743083 8096674 8104152 8167144 ...
 $ end     : int  1418149 1603163 1607399 1907729 2643618 4797326 6744180 8097832 8105133 8168238 ...
'data.frame':	2925 obs. of  3 variables:
 $ seqnames: chr  "chr1" "chr1" "chr1" "chr1" ...
 $ start   : int  992333